# 05 — 10% Limited-Data Baseline: Drone–Bird Micro-Doppler Classification

This notebook trains, recovers, and evaluates a compact CNN using the smallest balanced real-data subset: **575 bird segments and 575 drone segments**. It establishes the reference against which synthetic radar augmentation will later be evaluated.

The notebook defaults to **recovery mode**, so an existing successful experiment is loaded instead of overwritten.

## 1. Imports and Reproducibility

The random seed is fixed across Python, NumPy, and TensorFlow. On native Windows, TensorFlow 2.21 uses the CPU; the absence of a GPU is expected.

In [ ]:
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from pathlib import Path
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("TensorFlow deterministic operations enabled.")
except Exception as error:
    print("Deterministic operations unavailable:", error)

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Paths and Recovery Protection

`RETRAIN_MODEL` is deliberately set to `False`. Change it to `True` only when a new experiment is intentionally required.

In [ ]:
OFFICIAL_DATA_DIR = Path("../data/processed/official_split")
LIMITED_DATA_DIR = Path("../data/processed/limited_subsets")
OUTPUT_DIR = Path("../outputs/baseline_classification")
CHECKPOINT_DIR = Path("../checkpoints/baseline")

SELECTED_SUBSET = "10_percent"
RETRAIN_MODEL = False
RESULT_DIR = OUTPUT_DIR / f"{SELECTED_SUBSET}_seed_{RANDOM_SEED}"
CHECKPOINT_PATH = CHECKPOINT_DIR / f"baseline_{SELECTED_SUBSET}_seed_{RANDOM_SEED}.keras"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

required_data = [
    OFFICIAL_DATA_DIR / "X_train.npy", OFFICIAL_DATA_DIR / "y_train.npy",
    OFFICIAL_DATA_DIR / "X_validation.npy", OFFICIAL_DATA_DIR / "y_validation.npy",
    OFFICIAL_DATA_DIR / "X_test.npy", OFFICIAL_DATA_DIR / "y_test.npy",
    OFFICIAL_DATA_DIR / "metadata_test.csv",
    LIMITED_DATA_DIR / "indices_10_percent.npy"
]
missing_data = [path for path in required_data if not path.exists()]
if missing_data:
    raise FileNotFoundError("Missing files:\n" + "\n".join(map(str, missing_data)))

print("All required processed-data files were found.")
print("Result directory:", RESULT_DIR.resolve())
print("Checkpoint exists:", CHECKPOINT_PATH.exists())
print("Retraining enabled:", RETRAIN_MODEL)

## 3. Load the 10% Training Subset and Fixed Evaluation Sets

The balanced training subset contains 575 samples per class. Validation and test sets retain their official imbalanced class distributions. Bird is encoded as 0 and drone as 1.

In [ ]:
X_train_base = np.load(OFFICIAL_DATA_DIR / "X_train.npy", mmap_mode="r")
y_train_base = np.load(OFFICIAL_DATA_DIR / "y_train.npy", mmap_mode="r")
X_validation = np.load(OFFICIAL_DATA_DIR / "X_validation.npy", mmap_mode="r")
y_validation = np.load(OFFICIAL_DATA_DIR / "y_validation.npy")
X_test = np.load(OFFICIAL_DATA_DIR / "X_test.npy", mmap_mode="r")
y_test = np.load(OFFICIAL_DATA_DIR / "y_test.npy")
metadata_test = pd.read_csv(OFFICIAL_DATA_DIR / "metadata_test.csv")

subset_indices = np.load(LIMITED_DATA_DIR / "indices_10_percent.npy")
X_train = np.asarray(X_train_base[subset_indices], dtype=np.float32)[..., np.newaxis]
y_train = np.asarray(y_train_base[subset_indices], dtype=np.uint8)

print("Training:", X_train.shape, y_train.shape, np.bincount(y_train))
print("Validation:", X_validation.shape, y_validation.shape, np.bincount(y_validation))
print("Test:", X_test.shape, y_test.shape, np.bincount(y_test))
assert X_train.shape == (1150, 5, 150, 1)
assert np.array_equal(np.bincount(y_train), [575, 575])
assert np.isfinite(X_train).all()

In [ ]:
BATCH_SIZE = 64
train_dataset = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
                 .shuffle(len(y_train), seed=RANDOM_SEED, reshuffle_each_iteration=True)
                 .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
validation_dataset = (tf.data.Dataset.from_tensor_slices((X_validation[..., np.newaxis], y_validation))
                      .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
test_dataset = (tf.data.Dataset.from_tensor_slices((X_test[..., np.newaxis], y_test))
                .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
print("Training batches:", len(train_dataset))
print("Validation batches:", len(validation_dataset))
print("Test batches:", len(test_dataset))

## 4. Compact CNN Architecture

The model uses kernels elongated along the Doppler dimension, preserves all five range cells during pooling, and contains approximately 29,000 parameters to limit overfitting.

In [ ]:
from tensorflow.keras import layers, models, regularizers

def build_baseline_model(input_shape=(5, 150, 1)):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(16, (3, 7), padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D((1, 2)),
        layers.Conv2D(32, (3, 5), padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D((1, 2)),
        layers.Conv2D(64, (3, 3), padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D((1, 2)),
        layers.GlobalAveragePooling2D(),
        layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.30),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="roc_auc"),
                 tf.keras.metrics.AUC(name="pr_auc", curve="PR"),
                 tf.keras.metrics.Precision(name="precision"),
                 tf.keras.metrics.Recall(name="recall")]
    )
    return model

## 5. Recover or Intentionally Train the Model

In recovery mode, the saved checkpoint is loaded. Training occurs only after manually setting `RETRAIN_MODEL = True`. The test set is never used for early stopping or threshold selection.

In [ ]:
if RETRAIN_MODEL:
    random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); tf.random.set_seed(RANDOM_SEED)
    model = build_baseline_model()
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(CHECKPOINT_PATH, monitor="val_loss", save_best_only=True, verbose=1),
        tf.keras.callbacks.CSVLogger(RESULT_DIR / "training_log.csv")
    ]
    history = model.fit(train_dataset, validation_data=validation_dataset, epochs=50, callbacks=callbacks)
    pd.DataFrame(history.history).to_csv(RESULT_DIR / "training_history.csv", index=False)
else:
    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError("Checkpoint missing. Verify the path before enabling retraining.")
    model = tf.keras.models.load_model(CHECKPOINT_PATH)
    print("Existing checkpoint loaded; no training was performed.")

model.summary()

## 6. Training Behaviour

The original run stopped after epoch 14 and restored epoch 6, where validation loss was lowest. Training performance continued improving afterward while validation loss deteriorated, indicating overfitting.

In [ ]:
history_path = RESULT_DIR / "training_history.csv"
if history_path.exists():
    history_df = pd.read_csv(history_path)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, metric, title in zip(axes, ["loss", "accuracy", "roc_auc"],
                                  ["Binary Cross-Entropy Loss", "Accuracy", "ROC-AUC"]):
        ax.plot(history_df[metric], label="Training")
        ax.plot(history_df[f"val_{metric}"], label="Validation")
        ax.set_title(title); ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=0.2)
    plt.tight_layout(); plt.show()
else:
    print("Training history not found; the checkpoint can still be evaluated.")

## 7. Validation-Only Threshold Selection

At threshold 0.5, every validation sample was predicted as a drone. The model ranked the classes well but its probabilities were shifted upward. The threshold is therefore selected by maximum validation macro-F1, with balanced accuracy as a tie-breaker. It is locked before test evaluation.

In [ ]:
validation_probabilities = model.predict(validation_dataset, verbose=0).reshape(-1)
threshold_records = []
for threshold in np.linspace(0.01, 0.99, 199):
    pred = (validation_probabilities >= threshold).astype(np.uint8)
    threshold_records.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_validation, pred),
        "balanced_accuracy": balanced_accuracy_score(y_validation, pred),
        "macro_f1": f1_score(y_validation, pred, average="macro", zero_division=0),
        "bird_recall": np.mean(pred[y_validation == 0] == 0),
        "drone_recall": np.mean(pred[y_validation == 1] == 1)
    })
threshold_df = pd.DataFrame(threshold_records)
best = threshold_df.sort_values(["macro_f1", "balanced_accuracy"], ascending=False).iloc[0]
optimal_threshold = float(best["threshold"])
print("Selected threshold:", optimal_threshold)
display(best.to_frame().T.round(4))

**Original validation result.** The selected threshold was approximately **0.7623**, producing 0.8646 balanced accuracy and 0.8172 macro-F1. The default threshold produced zero bird recall and must not be used. Small numerical differences can occur if a different threshold grid is used.

## 8. Untouched Test-Set Evaluation

In [ ]:
# Prefer the precisely saved threshold when available.
saved_metrics_path = RESULT_DIR / "test_metrics.csv"
if saved_metrics_path.exists():
    saved_metrics = pd.read_csv(saved_metrics_path)
    optimal_threshold = float(saved_metrics.loc[0, "threshold"])

test_probabilities = model.predict(test_dataset, verbose=0).reshape(-1)
test_predictions = (test_probabilities >= optimal_threshold).astype(np.uint8)
report = classification_report(y_test, test_predictions, target_names=["bird", "drone"],
                               output_dict=True, zero_division=0)
test_metrics_df = pd.DataFrame([{
    "subset": SELECTED_SUBSET, "seed": RANDOM_SEED, "threshold": optimal_threshold,
    "accuracy": accuracy_score(y_test, test_predictions),
    "balanced_accuracy": balanced_accuracy_score(y_test, test_predictions),
    "macro_f1": f1_score(y_test, test_predictions, average="macro"),
    "bird_precision": report["bird"]["precision"],
    "bird_recall": report["bird"]["recall"], "bird_f1": report["bird"]["f1-score"],
    "drone_precision": report["drone"]["precision"],
    "drone_recall": report["drone"]["recall"], "drone_f1": report["drone"]["f1-score"],
    "roc_auc": roc_auc_score(y_test, test_probabilities)
}])
display(test_metrics_df.round(4))
print(classification_report(y_test, test_predictions, target_names=["bird", "drone"], digits=4))

In [ ]:
cm = confusion_matrix(y_test, test_predictions)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Bird", "Drone"],
            yticklabels=["Bird", "Drone"])
plt.title("Test Confusion Matrix — 10% Baseline")
plt.xlabel("Predicted target"); plt.ylabel("True target")
plt.tight_layout(); plt.show()

### Global Test Conclusion

The original run achieved **89.42% accuracy**, **84.97% balanced accuracy**, **80.82% macro-F1**, and **93.53% ROC-AUC**. It correctly classified 785/997 birds and 5,471/5,999 drones. Bird precision was lower (59.79%) because 528 drones were predicted as birds in the strongly imbalanced test set. Validation and test results were close, indicating reasonable segment-level generalization.

## 9. Sample-Level Result Table and Subtype Analysis

Subtype recall measures whether each original category was assigned to the correct broad class; it is not species- or model-level multiclass accuracy.

In [ ]:
test_results_df = metadata_test.copy()
test_results_df["true_binary_label"] = y_test
test_results_df["drone_probability"] = test_probabilities
test_results_df["predicted_binary_label"] = test_predictions
test_results_df["true_target_group"] = np.where(y_test == 1, "drone", "bird")
test_results_df["predicted_target_group"] = np.where(test_predictions == 1, "drone", "bird")
test_results_df["correct"] = y_test == test_predictions

subtype_results = (test_results_df.groupby(["true_target_group", "original_label"], observed=True)
    .agg(samples=("correct", "size"), correct_predictions=("correct", "sum"),
         classification_recall=("correct", "mean"),
         mean_drone_probability=("drone_probability", "mean"),
         median_drone_probability=("drone_probability", "median"))
    .reset_index().sort_values(["true_target_group", "classification_recall"]))
display(subtype_results.style.format({"classification_recall": "{:.2%}",
    "mean_drone_probability": "{:.4f}", "median_drone_probability": "{:.4f}"}))

### Subtype Conclusion

D1 initially appeared hardest at 70.20% recall, while D6 reached 99.70%. Seagull and black-headed gull recalls were 74.59% and 79.30%. Pigeon and raven had only four and one test samples, so their 0% recalls are not statistically interpretable. Later range analysis showed that the apparent D1 weakness is almost entirely a long-range effect.

## 10. Range-Stratified Evaluation

In [ ]:
RANGE_BINS = [0, 30, 50, 75, np.inf]
RANGE_LABELS = ["<30 m", "30–50 m", "50–75 m", ">75 m"]
test_results_df["range_bin"] = pd.cut(test_results_df["range_m"], RANGE_BINS,
                                           labels=RANGE_LABELS, right=False, include_lowest=True)

records = []
for label in RANGE_LABELS:
    group = test_results_df[test_results_df["range_bin"] == label]
    yt = group["true_binary_label"].to_numpy(); yp = group["predicted_binary_label"].to_numpy()
    prob = group["drone_probability"].to_numpy()
    records.append({"range_bin": label, "samples": len(group),
        "bird_samples": int(np.sum(yt == 0)), "drone_samples": int(np.sum(yt == 1)),
        "accuracy": accuracy_score(yt, yp),
        "balanced_accuracy": balanced_accuracy_score(yt, yp),
        "macro_f1": f1_score(yt, yp, average="macro"),
        "bird_recall": np.mean(yp[yt == 0] == 0),
        "drone_recall": np.mean(yp[yt == 1] == 1),
        "roc_auc": roc_auc_score(yt, prob), "mean_drone_probability": np.mean(prob)})
range_metrics = pd.DataFrame(records)
display(range_metrics.round(4))

In [ ]:
plot_df = range_metrics.melt(id_vars="range_bin",
    value_vars=["bird_recall", "drone_recall", "balanced_accuracy"],
    var_name="metric", value_name="score")
plt.figure(figsize=(11, 6))
sns.barplot(data=plot_df, x="range_bin", y="score", hue="metric", order=RANGE_LABELS)
plt.axhline(0.5, color="black", linestyle="--", linewidth=1, label="50% reference")
plt.ylim(0, 1.05); plt.xlabel("Target range interval"); plt.ylabel("Score")
plt.title("Test Performance by Target Range — 10% Baseline")
plt.grid(axis="y", alpha=0.25); plt.tight_layout(); plt.show()

### Range Conclusion

The classifier exhibits a strong range-dependent probability shift. Below 30 m, bird recall was 49.52% and drone recall 99.13%. Beyond 75 m, bird recall rose to 98.04% while drone recall fell to 51.79%. Performance was most balanced between 30 m and 75 m. Ordinary accuracy was misleading at the range extremes because drones strongly outnumbered birds.

## 11. Joint Subtype–Range Analysis

In [ ]:
subtype_range_performance = (test_results_df
    .groupby(["true_target_group", "original_label", "range_bin"], observed=True)
    .agg(samples=("correct", "size"), correct_predictions=("correct", "sum"),
         recall=("correct", "mean"),
         mean_drone_probability=("drone_probability", "mean"))
    .reset_index())
subtype_range_performance["sufficient_support"] = subtype_range_performance["samples"] >= 20
display(subtype_range_performance.style.format({"recall": "{:.2%}",
                                                   "mean_drone_probability": "{:.4f}"}))

### Joint Diagnostic Conclusion

D1 was classified nearly perfectly below 75 m but achieved only 47.04% recall beyond 75 m. Of its 298 errors, 295 occurred beyond 75 m. D1 also contributed 557/672 long-range drone samples, amplifying the global long-range failure. Within both seagull and black-headed gull, bird recall improved consistently with range, confirming that the range effect is not solely a subtype-composition artifact. Synthetic augmentation should therefore target long-range drones and short-range gulls while preserving subtype diversity.

## 12. Save or Restore All Experiment Artifacts

The following cell writes reproducible tables without retraining or overwriting the model checkpoint.

In [ ]:
test_metrics_df.to_csv(RESULT_DIR / "test_metrics.csv", index=False)
test_results_df.to_csv(RESULT_DIR / "test_predictions.csv", index=False)
subtype_results.to_csv(RESULT_DIR / "subtype_metrics.csv", index=False)
range_metrics.to_csv(RESULT_DIR / "range_metrics.csv", index=False)
subtype_range_performance.to_csv(RESULT_DIR / "subtype_range_metrics.csv", index=False)

configuration = {
    "subset": SELECTED_SUBSET, "random_seed": RANDOM_SEED,
    "training_samples": int(len(y_train)),
    "bird_training_samples": int(np.sum(y_train == 0)),
    "drone_training_samples": int(np.sum(y_train == 1)),
    "decision_threshold": float(optimal_threshold),
    "threshold_selection": "Maximum validation macro-F1 with balanced-accuracy tie-break",
    "test_metrics": {k: (v.item() if isinstance(v, np.generic) else v)
                     for k, v in test_metrics_df.iloc[0].to_dict().items()}
}
with open(RESULT_DIR / "experiment_config.json", "w", encoding="utf-8") as file:
    json.dump(configuration, file, indent=4)
print("Experiment artifacts saved to:", RESULT_DIR.resolve())

# Final Conclusion

Using only **575 real samples per class**, the compact CNN learned discriminative Doppler structure and achieved a strong segment-level baseline. Its principal limitation is not uniform classification failure but a range-associated probability shift: short-range birds tend toward drone predictions and long-range drones tend toward bird predictions.

This experiment is the reference point for the project. The next controlled stage trains the identical architecture on the 25%, 50%, and 100% balanced real-data subsets. Synthetic augmentation will later be considered beneficial only if it improves macro-F1, balanced accuracy, bird performance, and range robustness relative to the corresponding real-only baseline.

**Limitation:** the official partition is segment-level, so sessions can contribute to multiple splits. A later session-independent experiment is necessary before making strong claims about generalization to unseen measurement sessions.